# Eshmun Supervised Fine-Tuning — LoRA

SFT on `khairi/eshmun-instructions` starting from `khairi/Eshmun-0.3B-CPT-LoRA`.

**Strategy:**
1. Load base model and merge the CPT-LoRA adapter (frozen base becomes the SFT starting point)
2. Inject fresh SFT-LoRA adapters into attention and FFN layers
3. The `SFTCollator` applies the chat template **on the fly**: formats `(Input, Output)` pairs
   into user/assistant turns, tokenizes, and masks user tokens from the loss

**Pipeline:**
1. Install dependencies
2. Login to HuggingFace Hub
3. Imports
4. Configuration
5. Load model — merge CPT-LoRA, inject SFT-LoRA
6. Dataset
7. SFT data collator
8. Training
9. Save and push to Hub

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/abidikhairi/eshmun.git
!pip install -q datasets transformers accelerate peft

## 2. Login to HuggingFace Hub

In [ ]:
from huggingface_hub import login as hf_login

hf_login()  # paste your HF write-access token when prompted

## 3. Imports

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

In [ ]:
BASE_MODEL_ID  = "khairi/Eshmun-0.3B-Base"        # base weights (pre-CPT)
CPT_LORA_ID    = "khairi/Eshmun-0.3B-CPT-LoRA"    # CPT adapter to merge in
DATASET_ID     = "khairi/eshmun-instructions"
HF_REPO_ID     = "khairi/Eshmun-0.3B-SFT"         # SFT adapter pushed here
OUTPUT_DIR     = "/tmp/eshmun-0.3b-sft"

MAX_SEQ_LEN = 512

# LoRA hyperparameters
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# Training hyperparameters
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8        # effective batch = 32
NUM_TRAIN_EPOCHS            = 3
LEARNING_RATE               = 2e-4
WARMUP_STEPS                = 100
LOGGING_STEPS               = 50
SAVE_STEPS                  = 500

## 5. Model & tokenizer

1. Load `BASE_MODEL_ID`
2. Attach the CPT-LoRA adapter from `CPT_LORA_ID` and merge it into the base weights — the merged
   model is the SFT starting point with no leftover adapter bookkeeping
3. Wrap with fresh SFT-LoRA adapters

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CPT_LORA_ID, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float32,
)

# Merge CPT-LoRA into base weights, then discard adapter scaffolding
model = PeftModel.from_pretrained(base_model, CPT_LORA_ID)
model = model.merge_and_unload()

n_params = sum(p.numel() for p in model.parameters())
print(f"Merged model: {n_params / 1e6:.1f}M parameters")

## 6. SFT-LoRA setup

Fresh LoRA adapters injected on top of the merged CPT weights.
All base weights stay frozen; only the adapter matrices are updated.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Dataset

Raw strings only — no upfront tokenization. The `SFTCollator` handles everything at batch time.

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.select_columns(["Input", "Output"])

print(dataset)
print(dataset[0])

## 8. SFT data collator

Applies the chat template **on the fly** at batch time:

- Formats each `(Input, Output)` pair as `user` / `assistant` turns
- Tokenizes the full conversation and the prompt-only prefix to determine the prompt length
- Sets `labels = -100` for all prompt tokens so the loss is computed only on assistant tokens
- Pads the batch to the longest sequence

In [ ]:
@dataclass
class SFTCollator:
    tokenizer: PreTrainedTokenizerBase
    max_length: int = 512

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        all_input_ids: list[list[int]] = []
        all_attention_masks: list[list[int]] = []
        all_labels: list[list[int]] = []

        for f in features:
            messages = [
                {"role": "user",      "content": f["Input"]},
                {"role": "assistant", "content": f["Output"]},
            ]

            # Full conversation: prompt + response
            full_ids: list[int] = self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
                max_length=self.max_length,
                truncation=True,
            )

            # Prompt-only prefix — tells us where the assistant response starts
            prompt_ids: list[int] = self.tokenizer.apply_chat_template(
                messages[:1],
                tokenize=True,
                add_generation_prompt=True,
                max_length=self.max_length,
                truncation=True,
            )

            prompt_len = len(prompt_ids)
            # Mask prompt tokens; compute loss only on assistant tokens
            labels = [-100] * prompt_len + full_ids[prompt_len:]

            all_input_ids.append(full_ids)
            all_attention_masks.append([1] * len(full_ids))
            all_labels.append(labels)

        # Pad to the longest sequence in the batch (right-pad)
        max_len = max(len(ids) for ids in all_input_ids)
        pad_id = self.tokenizer.pad_token_id

        for i in range(len(all_input_ids)):
            pad_len = max_len - len(all_input_ids[i])
            all_input_ids[i]      += [pad_id] * pad_len
            all_attention_masks[i] += [0]      * pad_len
            all_labels[i]          += [-100]   * pad_len

        return {
            "input_ids":      torch.tensor(all_input_ids,      dtype=torch.long),
            "attention_mask": torch.tensor(all_attention_masks, dtype=torch.long),
            "labels":         torch.tensor(all_labels,         dtype=torch.long),
        }


collator = SFTCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)

### Collator sanity check

In [ ]:
batch = collator([dataset[0], dataset[1]])
print("input_ids shape :", batch["input_ids"].shape)
print("labels shape    :", batch["labels"].shape)
n_masked   = (batch["labels"][0] == -100).sum().item()
n_unmasked = (batch["labels"][0] != -100).sum().item()
print(f"labels[0]: {n_masked} masked (prompt), {n_unmasked} active (assistant)")

## 9. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    fp16=False,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collator,
)

trainer.train()

## 10. Save and push to Hub

`model.save_pretrained` saves only the **SFT adapter weights** (not the merged base).
Load it back with:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("khairi/Eshmun-0.3B-Base", trust_remote_code=True)
cpt  = PeftModel.from_pretrained(base, "khairi/Eshmun-0.3B-CPT-LoRA")
merged = cpt.merge_and_unload()
model  = PeftModel.from_pretrained(merged, "khairi/Eshmun-0.3B-SFT")
```

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to {OUTPUT_DIR}")

trainer.push_to_hub(commit_message="sft lora checkpoint")
print(f"SFT adapter pushed to https://huggingface.co/{HF_REPO_ID}")